* LeNet is the first full CNN architecture in this path.
* It combines the pieces from Chapter 7 into one representation pipeline:
> * convolution for local feature extraction, activation for nonlinearity,
> * pooling for spatial summarization,
> * flattening for vector classification,
> * and dense layers for class logits.

# How to use this notebook

* Run the notebook from top to bottom.

* Every code block is designed to be cloud-runnable and self-contained inside this notebook.

* The drills are intentionally small: predict the shape or behavior first, run the cell, then read the assertion as the contract you must understand.

# You are done when you can

- explain LeNet as a staged representation pipeline
- trace LeNet tensor shapes layer by layer
- count parameters by layer
- run one synthetic classification training step
- debug the common flatten-to-linear size mismatch

In [4]:
import torch
from torch import nn

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def make_lenet():
    return nn.Sequential(
        nn.Conv2d(1, 6, kernel_size=5, padding=2), # Taking 1 input channel @ 6 feature maps with a kernel_size of 5*5 and 2x padding, or floor((28 + 2×2 - 5) / 1) + 1 = 28 for the shape of (batch, 6, 28, 28)
        nn.Sigmoid(),                              # Transform large negatives -> clsoe to 0, 0 -> 0.5, large positive -> close to 1
        nn.AvgPool2d(kernel_size=2, stride=2),     # output = floor((28 - 2) / 2) + 1 = 13 + 1 = 14, for the shape of (batch, 6, 14, 14)
        nn.Conv2d(6, 16, kernel_size=5),           # output = floor((14 - 5) / 1) + 1 = 9 + 1 = 10, for the shape of (batch, 16, 10, 10)
        nn.Sigmoid(),
        nn.AvgPool2d(kernel_size=2, stride=2),     # output = floor((10 - 2) / 2) + 1 = 4 + 1 = 5, for the shape of (batch, 16, 5, 5)
        nn.Flatten(),                              # Flattens shape to (batch, 16 * 5 *5) or (batch, 400)
        nn.Linear(16 * 5 * 5, 120),                # Shape of (batch, 400) @ linear.weight.T of (400, 120) -> (batch, 120)
        nn.Sigmoid(),
        nn.Linear(120, 84),                        # Shape of (batch, 120) @ linear.weight.T of (120, 84) -> (batch, 84)
        nn.Sigmoid(),
        nn.Linear(84, 10),                         # Shape of (batch, 84) @ linear.weight.T of (84, 10) -> (batch, 10)
    )

def trace_shapes(net, X):
    rows = []
    current = X
    for idx, layer in enumerate(net):
        current = layer(current)
        rows.append((idx, type(layer).__name__, shape(current)))
    return rows

# 7.6.0 The Problem This Notebook Solves

The previous notebooks studied CNN pieces separately. LeNet is where those pieces become an end-to-end classifier.

The representation story is:

```text
image pixels
-> local low-level feature maps
-> smaller pooled feature maps
-> richer feature maps
-> smaller pooled feature maps
-> flattened feature vector
-> class logits
```

This is not just a list of layers. It is a sequence of changing representation types:

- early tensors preserve spatial structure
- **channels increase as learned feature variety increases**
- **spatial size shrinks as the model summarizes location**
- flattening converts spatial feature maps into a vector
- dense layers turn that vector into class scores

This notebook does not try to get real accuracy. That belongs later.


The rigorous goal is to make the architecture auditable: every shape, parameter count, loss, gradient, and update should make sense.

# 7.6.1 Build LeNet

The original LeNet used sigmoid-style activations and average pooling.

Modern CNNs often use ReLU, batch normalization, residual connections, and different downsampling blocks.

LeNet is still useful because it is small enough to audit completely.

Read the model as a contract:

```text
input: batch, 1 channel, 28 height, 28 width
output: batch, 10 class scores
```

The final output values are logits. They are raw class scores, not probabilities.

In [3]:
net = make_lenet()
X = torch.zeros(2, 1, 28, 28)
Y = net(X)

print("output shape:", shape(Y))
print(net)

assert shape(Y) == (2, 10)

output shape: (2, 10)
Sequential(
  (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (1): Sigmoid()
  (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (3): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (4): Sigmoid()
  (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (6): Flatten(start_dim=1, end_dim=-1)
  (7): Linear(in_features=400, out_features=120, bias=True)
  (8): Sigmoid()
  (9): Linear(in_features=120, out_features=84, bias=True)
  (10): Sigmoid()
  (11): Linear(in_features=84, out_features=10, bias=True)
)


# 7.6.2 Trace Shapes Layer by Layer

Shape tracing is the most important LeNet skill. Without it, the first dense layer is a guess.

The expected spatial story is:

```text
28 by 28 input
Conv5 padding2 -> 28 by 28
AvgPool2 stride2 -> 14 by 14
Conv5 no padding -> 10 by 10
AvgPool2 stride2 -> 5 by 5
Flatten with 16 channels -> 16 * 5 * 5 = 400 features
```
* The first dense (linear) layer only works because its input feature count is 400.
* If any earlier padding, kernel, or stride changes, that number changes too.

The trace is a form of architectural proof. It shows that the tensor produced by each stage is the tensor the next stage expects.

In [5]:
net = make_lenet()
rows = trace_shapes(net, torch.zeros(1, 1, 28, 28))

for idx, name, out_shape in rows:
    print(f"{idx:2d} {name:10s} -> {out_shape}")

assert rows[0][2] == (1, 6, 28, 28)
assert rows[2][2] == (1, 6, 14, 14)
assert rows[3][2] == (1, 16, 10, 10)
assert rows[5][2] == (1, 16, 5, 5)
assert rows[6][2] == (1, 400)
assert rows[-1][2] == (1, 10)

 0 Conv2d     -> (1, 6, 28, 28)
 1 Sigmoid    -> (1, 6, 28, 28)
 2 AvgPool2d  -> (1, 6, 14, 14)
 3 Conv2d     -> (1, 16, 10, 10)
 4 Sigmoid    -> (1, 16, 10, 10)
 5 AvgPool2d  -> (1, 16, 5, 5)
 6 Flatten    -> (1, 400)
 7 Linear     -> (1, 120)
 8 Sigmoid    -> (1, 120)
 9 Linear     -> (1, 84)
10 Sigmoid    -> (1, 84)
11 Linear     -> (1, 10)


# 7.6.3 Count Parameters by Layer

Parameter counting tells you where model capacity lives.

For convolution:

```text
output channels * input channels * kernel height * kernel width
plus bias values
```

For dense (linear) layers:

```text
output features * input features
plus bias values
```

* LeNet's dense layers contain many parameters because flattening creates a 400-feature vector and then connects it densely.
* This is one reason modern CNNs often use global average pooling or other designs to reduce dense classifier size.

Counting parameters is not performance worship.

It is a way to reason about capacity, memory, overfitting risk, and where updates will occur.

In [6]:
net = make_lenet()
total = 0

for idx, layer in enumerate(net):
    params = sum(p.numel() for p in layer.parameters())
    if params:
        print(f"{idx:2d} {type(layer).__name__:10s} params={params}")
    total += params

print("total parameters:", total)

expected = (
    6 * 1 * 5 * 5 + 6                   # Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    + 16 * 6 * 5 * 5 + 16               # Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
    + 120 * (16 * 5 * 5) + 120          # Linear(in_features=400, out_features=120, bias=True)
    + 84 * 120 + 84                     # Linear(in_features=120, out_features=84, bias=True)
    + 10 * 84 + 10                      # Linear(in_features=84, out_features=10, bias=True)
)

assert total == expected

 0 Conv2d     params=156
 3 Conv2d     params=2416
 7 Linear     params=48120
 9 Linear     params=10164
11 Linear     params=850
total parameters: 61706


# 7.6.4 One Synthetic Training Step

This cell is a wiring proof, not an accuracy experiment.

It checks the end-to-end training contract:

```text
image batch -> logits
logits plus integer labels -> scalar classification loss
loss.backward -> gradients on parameters
optimizer.step -> parameter values change
```

* The labels are synthetic and the images are random, so the loss value has no real-world meaning.
* That restraint is intentional.
* A full dataset experiment would introduce data loading, metrics, train/test splits, regularization, and comparison baselines.
* Here we isolate the CNN mechanics.

This is the chapter-level version of rigor: prove the model is trainable before asking whether it is useful.

In [7]:
torch.manual_seed(0)
net = make_lenet()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(net.parameters(), lr=0.1)

X = torch.randn(4, 1, 28, 28)
y = torch.tensor([0, 1, 2, 3])

first_weight = net[0].weight
before = first_weight.detach().clone()

logits = net(X)
loss = loss_fn(logits, y)

optimizer.zero_grad()
loss.backward()
optimizer.step()

after = first_weight.detach().clone()

print("logits shape:", shape(logits)) # (4, 10) since batch_size from X = 4 and the final dense (linear) layer output features = 10
print("loss:", float(loss.detach()))
print("first cov grad shape:", shape(first_weight.grad)) # Shape of (6, 1, 5, 5) given layer as Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
print("first conv changed:", not torch.allclose(before, after))

assert shape(logits) == (4, 10)
assert loss.ndim == 0
assert shape(first_weight.grad) == shape(first_weight)
assert not torch.allclose(before, after)

logits shape: (4, 10)
loss: 2.318662166595459
first cov grad shape: (6, 1, 5, 5)
first conv changed: True


# 7.6.5 Prediction Mechanics

The network returns logits: raw class scores.
* `CrossEntropyLoss` expects logits because it internally handles the softmax-like normalization in a numerically stable way.
* For prediction, `argmax(dim=1)` selects the highest-scoring class for each example.
* This does not require converting to probabilities first because softmax preserves score order.

The axis matters:

```text
dim=0 would compare across batch examples
dim=1 compares across classes for each example
```

That is why the predicted label tensor has shape `(batch,)`.

In [8]:
with torch.no_grad():
    logits = net(torch.randn(3, 1, 28, 28))
    predicted = logits.argmax(dim=1) # Given the shape of (3, 10), find the maximum logit index per row (armgax across columns)

print("logits shape:", shape(logits)) # (3, 10) since batch_size from X = 3 and the final dense (linear) layer output features = 10
print("predicted labels:", predicted)

assert shape(logits) == (3, 10)
assert shape(predicted) == (3, )

logits shape: (3, 10)
predicted labels: tensor([3, 3, 3])


# 7.6.6 Break It Deliberately: Wrong Flattened Size

This is one of the most common CNN mistakes.
* The convolution and pooling stack produces `(batch, 16, 5, 5)`.
* `Flatten` converts that to `(batch, 400)`.
* If the next dense layer expects 256 features instead of 400, the model fails at matrix multiplication.

The conceptual mistake is losing track of the representation handoff:

```text
spatial feature map -> flattened vector -> dense layer
```
The dense layer does not know the spatial history, as it only receives a vector of a certain length.

The architecture must handle the length correctly.

In [9]:
bad_net = nn.Sequential(
    nn.Conv2d(1, 6, kernel_size=5, padding=2),
    nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Conv2d(6, 16, kernel_size=5),
    nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Flatten(),
    nn.Linear(16 * 4 * 4, 120), # wrong: actual flattened size is 16 * 5 * 5
)

try:
    bad_net(torch.zeros(1, 1, 28, 28))
except RuntimeError as err:
    print(type(err).__name__)
    print(str(err).splitlines()[0])
else:
    raise AssertionError("The wrong flattened size should have failed.")

RuntimeError
mat1 and mat2 shapes cannot be multiplied (1x400 and 256x120)


# 7.6.7 What This Chapter Does Not Do

This chapter does not run a full dataset experiment, compare MLP against CNN accuracy, tune regularization, save curves, or write a custom optimizer.

Those belong after Chapters 5-7 are mechanically solid.

The chapter goal is conceptual and mechanical fluency:

- why convolution is a good bias for image-like data
- how channels and spatial dimensions flow
- how pooling trades detail for summary
- why LeNet's dense layer expects 400 features
- how logits, loss, gradients, and updates connect in one CNN step

That is the handoff: after this chapter, a larger CNN training exercise can focus on data, evaluation, baselines, and training discipline because the layer mechanics are no longer mysterious.

# 7.6 Checkpoint

Answer these before moving on.

Short markdown answers in the notebook are enough; the chapter does not need a separate notes file.

1. How does LeNet change the representation from image grid to class scores?
> LeNet progressively transforms the image grid into feature maps using convolution and pooling, then flattens those feature maps into a vector and uses dense layers to produce 10 class logits

2. Why is the flattened LeNet feature size `16 * 5 * 5` for 28 by 28 inputs?
> The second convolution produces 16 channels, and the pooling layers reduce the spatial size to 5 × 5, giving `16 * 5 * 5 = 400` features

3. Why do dense layers contain many of LeNet's parameters?
> Dense layers contain many parameters because every input feature is connected to every output feature. For example, the 400-to-120 layer alone has 400 × 120 weights plus 120 biases

4. What is the difference between logits and probabilities?
> Logits are raw, unnormalized class scores, while probabilities are normalized scores that sum to 1. `CrossEntropyLoss` expects logits and internally performs the appropriate normalization

5. What did the synthetic training step prove, and what did it not prove?
> The synthetic step proved that the CNN can perform a complete forward pass, compute a loss, backpropagate gradients, and update its parameters. It did not prove that the model learns useful features or achieves good accuracy because the images and labels were random

6. Why does the wrong dense-layer input size fail only after `Flatten`?
> The convolution and pooling layers produce `(batch, 16, 5, 5)`, which Flatten converts to `(batch, 400)`. The `Linear` layer requires its input feature count to match its `in_features`. Therefore, if it expects 256 instead of 400, the matrix multiplication fails at the `Linear` layer immediately after `Flatten`
